<a target="_blank" href="https://colab.research.google.com/github/binsue0/.github/blob/main/ML_day1/1_3_2_)LogisticRegression.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Logistic Regression

### Reference
Hands-on Machine Learning, https://datascienceschool.net/

이름은 회귀(regression)인데 하는 일은 **분류**입니다. 왜 그런지부터 따라가 봅니다. 

- 왜 분류에 선형회귀를 쓰면 안 되는가
- 시그모이드 · 오즈 · 로짓
- 계수 해석 (오즈비)
- 비용함수: 로그 손실
- 결정 경계와 규제(C)
- 다중 클래스: 소프트맥스 회귀

## 1. 선형회귀로 분류를 하면?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

iris = load_iris()

# 꽃잎 너비 하나로 '버지니카 품종인가'를 맞혀 봅니다.
X = iris.data[:, 3:]                    # petal width
y = (iris.target == 2).astype(int)      # 버지니카면 1, 아니면 0

X[:5].ravel(), y[:5]

In [ ]:
# 일단 선형회귀(직선)를 그어봅니다.
from sklearn.linear_model import LinearRegression

lin = LinearRegression().fit(X, y)
grid = np.linspace(0, 3, 1000).reshape(-1, 1)

plt.figure(figsize=(7, 3.5))
plt.scatter(X, y, alpha=0.5)
plt.plot(grid, lin.predict(grid), 'r', linewidth=2)
plt.axhline(1, color='gray', linestyle=':')
plt.axhline(0, color='gray', linestyle=':')
plt.xlabel('Petal width (cm)')
plt.ylabel('y')
plt.show()

print('예측값 범위:', lin.predict(grid).min().round(2), '~', lin.predict(grid).max().round(2))

확률이라고 하기엔 이상하죠? **0보다 작고 1보다 큰 값**이 나옵니다.
직선은 위아래로 끝없이 뻗기 때문에 애초에 확률을 표현할 수 없습니다.

그래서 직선의 출력을 **0~1 사이로 눌러주는 함수**를 하나 씌웁니다.

## 2. 시그모이드 · 오즈 · 로짓

$$\hat{p} = \sigma(z), \qquad z = \beta_0 + \beta_1 x, \qquad \sigma(z) = \frac{1}{1+e^{-z}}$$

이 식을 뒤집으면 로지스틱 회귀의 정체가 보입니다.

$$\underbrace{\log \frac{p}{1-p}}_{\text{로짓(logit)}} = \beta_0 + \beta_1 x$$

즉 **확률 자체가 아니라 로짓을 선형으로 모델링**하는 것입니다. 그래서 이름에 회귀가 붙습니다.

- **오즈(odds)** $= \dfrac{p}{1-p}$ : 일어날 확률 대 일어나지 않을 확률의 비. (0.75면 오즈는 3, 즉 "3대 1")
- **로짓(logit)**: 오즈에 로그를 씌운 값. $-\infty \sim \infty$ 를 오갑니다.

In [ ]:
z = np.linspace(-8, 8, 300)
sigmoid = 1 / (1 + np.exp(-z))

p = np.linspace(0.001, 0.999, 300)
logit = np.log(p / (1 - p))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].plot(z, sigmoid, 'b', linewidth=2)
axes[0].axhline(0.5, color='r', linestyle=':')
axes[0].set_xlabel('z')
axes[0].set_ylabel('sigmoid(z) = probability')
axes[0].set_title('sigmoid: z -> probability')
axes[0].grid(True)

axes[1].plot(p, logit, 'g', linewidth=2)
axes[1].axhline(0, color='r', linestyle=':')
axes[1].set_xlabel('probability p')
axes[1].set_ylabel('logit = log(p/(1-p))')
axes[1].set_title('logit: probability -> z')
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 3. 학습하고 결정 경계 보기

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(random_state=42)
log_reg.fit(X, y)

y_proba = log_reg.predict_proba(grid)

# 확률이 0.5를 넘는 지점 = 결정 경계(decision boundary)
boundary = grid[y_proba[:, 1] >= 0.5][0, 0]
boundary

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.plot(X[y == 0], y[y == 0], 'bs', label='not virginica')
plt.plot(X[y == 1], y[y == 1], 'g^', label='virginica')
plt.plot(grid, y_proba[:, 1], 'g-', linewidth=2, label='P(virginica)')
plt.plot(grid, y_proba[:, 0], 'b--', linewidth=2, label='P(not virginica)')
plt.axvline(boundary, color='k', linestyle=':', linewidth=2)
plt.text(boundary + 0.03, 0.5, f'decision boundary\n{boundary:.2f} cm', fontsize=10)
plt.xlabel('Petal width (cm)')
plt.ylabel('Probability')
plt.legend(loc='center left')
plt.show()

In [ ]:
# 확률과 판정을 직접 확인합니다.
for w in [1.5, 1.7, 2.0]:
    p_hat = log_reg.predict_proba([[w]])[0, 1]
    print(f'꽃잎 너비 {w}cm -> 버지니카일 확률 {p_hat:.3f} -> 판정 {log_reg.predict([[w]])[0]}')

## 4. 계수를 어떻게 읽나: 오즈비(odds ratio)

선형회귀에서는 "$x$가 1 늘면 $y$가 $\beta$만큼 는다"였습니다.
로지스틱 회귀에서는 **로그 오즈**가 $\beta$만큼 늘어납니다. 로그를 벗기면 곱하기가 됩니다.

$$\text{오즈} \times e^{\beta} \qquad (\text{오즈비, odds ratio})$$

In [ ]:
beta = log_reg.coef_[0, 0]

print('계수(beta)  :', beta.round(3))
print('절편        :', log_reg.intercept_[0].round(3))
print('오즈비 e^beta:', np.exp(beta).round(1))

꽃잎 너비가 **1cm 커지면 버지니카일 오즈가 약 76배**가 된다는 뜻입니다.
(확률이 76배가 되는 게 아닙니다. 확률은 1을 넘을 수 없으니까요.)

In [ ]:
# 직접 확인해 봅시다. 1.5cm와 2.5cm의 오즈를 비교합니다.
p1 = log_reg.predict_proba([[1.5]])[0, 1]
p2 = log_reg.predict_proba([[2.5]])[0, 1]

odds1 = p1 / (1 - p1)
odds2 = p2 / (1 - p2)

print(f'1.5cm 확률 {p1:.3f} -> 오즈 {odds1:.3f}')
print(f'2.5cm 확률 {p2:.3f} -> 오즈 {odds2:.3f}')
print('오즈의 비 :', (odds2 / odds1).round(1))

## 5. 비용함수: 로그 손실(log loss)

선형회귀는 MSE를 최소화했습니다. 로지스틱 회귀는 **로그 손실**(= 크로스 엔트로피)을 씁니다.

$$J(\boldsymbol\beta) = -\frac{1}{n}\sum_{i=1}^{n}
\Big[\, y_i \log \hat{p}_i + (1-y_i)\log(1-\hat{p}_i) \,\Big]$$

정답이 1인데 확률을 0에 가깝게 예측하면 벌점이 **무한대로** 커집니다.
"자신 있게 틀리는 것"을 강하게 벌주는 셈이죠.

In [ ]:
p_hat = np.linspace(0.001, 0.999, 300)

plt.figure(figsize=(7, 3.5))
plt.plot(p_hat, -np.log(p_hat), 'g', linewidth=2, label='true y = 1')
plt.plot(p_hat, -np.log(1 - p_hat), 'b--', linewidth=2, label='true y = 0')
plt.xlabel('predicted probability')
plt.ylabel('loss')
plt.ylim(0, 5)
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# sklearn으로 직접 계산해 보기
from sklearn.metrics import log_loss

print('로그 손실:', log_loss(y, log_reg.predict_proba(X)))

MSE를 쓰지 않는 이유는, 시그모이드에 MSE를 씌우면 비용함수가 울퉁불퉁해져서(non-convex)
경사하강법이 지역 최소값에 빠질 수 있기 때문입니다. 로그 손실은 **볼록(convex)** 이라 항상 최적해를 찾습니다.

## 6. 특성이 2개일 때의 결정 경계

In [ ]:
# 꽃잎 길이와 너비를 함께 사용합니다.
X2 = iris.data[:, 2:4]

log_reg2 = LogisticRegression(C=1e10, random_state=42).fit(X2, y)

print('계수:', log_reg2.coef_[0].round(2), '절편:', log_reg2.intercept_[0].round(2))

In [ ]:
def plot_boundary(model, X, y, ax=None, title=''):
    ax = ax or plt.gca()
    xx, yy = np.meshgrid(np.linspace(2.5, 7.5, 300), np.linspace(0.5, 2.8, 300))
    proba = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

    cs = ax.contour(xx, yy, proba, levels=[0.1, 0.3, 0.5, 0.7, 0.9], cmap='brg')
    ax.clabel(cs, inline=1, fontsize=9)
    ax.plot(X[y == 0, 0], X[y == 0, 1], 'bs', markersize=5)
    ax.plot(X[y == 1, 0], X[y == 1, 1], 'g^', markersize=5)
    ax.set_xlabel('Petal length')
    ax.set_ylabel('Petal width')
    ax.set_title(title)

plt.figure(figsize=(8, 4))
plot_boundary(log_reg2, X2, y, title='decision boundary (0.5 line) and probability contours')
plt.show()

가운데 0.5 선이 결정 경계입니다. **직선**이라는 점에 주목하세요.
로지스틱 회귀는 확률은 곡선으로 주지만, 경계 자체는 선형입니다.

## 7. 규제(regularization)와 C

sklearn의 `LogisticRegression`은 **기본적으로 L2 규제가 켜져 있습니다.**
`C`는 규제의 세기를 반대로 나타내는 값입니다. (릿지의 alpha와 반대 방향)

- **C가 작다** → 규제가 세다 → 계수가 작아지고 경계가 완만해짐
- **C가 크다** → 규제가 약하다 → 계수가 커지고 데이터에 딱 맞춤

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, c in zip(axes, [0.01, 1, 100]):
    m = LogisticRegression(C=c, max_iter=1000, random_state=42).fit(X2, y)
    plot_boundary(m, X2, y, ax=ax, title=f'C = {c}  (coef {m.coef_[0].round(2)})')

plt.tight_layout()
plt.show()

C가 작을 때는 계수가 작아 확률 등고선이 넓게 퍼지고(자신 없어 함),
C가 클수록 등고선이 촘촘해집니다(자신 있게 단정함).

## 8. 다중 클래스: 소프트맥스 회귀

클래스가 3개 이상이면 각 클래스마다 점수를 내고 **소프트맥스**로 확률을 만듭니다.

$$\hat{p}_k = \frac{\exp(z_k)}{\sum_{j} \exp(z_j)}$$

확률의 합은 항상 1이 되고, 가장 큰 확률의 클래스로 판정합니다.

In [ ]:
y_all = iris.target        # 0=setosa, 1=versicolor, 2=virginica

softmax_reg = LogisticRegression(max_iter=1000, random_state=42).fit(X2, y_all)

print('정확도:', round(softmax_reg.score(X2, y_all), 3))

In [ ]:
# 세 품종의 결정 영역
xx, yy = np.meshgrid(np.linspace(0.5, 7.5, 400), np.linspace(0, 3, 400))
Z = softmax_reg.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 4))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
for k, marker in enumerate(['s', '^', 'o']):
    plt.plot(X2[y_all == k, 0], X2[y_all == k, 1], marker,
             markersize=5, label=iris.target_names[k])
plt.xlabel('Petal length')
plt.ylabel('Petal width')
plt.legend()
plt.show()

In [ ]:
# 확률을 직접 봅니다. 합은 1입니다.
sample = [[5, 2]]

print(iris.target_names)
print('확률 :', softmax_reg.predict_proba(sample).round(3))
print('판정 :', iris.target_names[softmax_reg.predict(sample)[0]])

## 정리

| | 선형회귀 | 로지스틱 회귀 |
|---|---|---|
| 출력 | 실수 | 0~1 확률 |
| 모델 | $y = \beta_0 + \beta_1 x$ | $\log\frac{p}{1-p} = \beta_0 + \beta_1 x$ |
| 비용함수 | MSE | 로그 손실(크로스 엔트로피) |
| 계수 해석 | $x$ 1 증가 → $y$ 가 $\beta$ 증가 | $x$ 1 증가 → 오즈가 $e^{\beta}$ 배 |
